# BiLSTM_MHA_internal

Internal BiLSTM-MHA analysis notebook. This notebook defines the attention-enhanced recurrent comparator described in the manuscript and loads the internal comparison table.

In [ ]:
from pathlib import Path
import csv
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
features_path = ROOT / 'data' / 'Data_6kGoc' / 'features.csv'
with features_path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    descriptor_cols = [c for c in reader.fieldnames if c not in {'ID', 'Label', 'Sequence'}]
print('Descriptor columns:', len(descriptor_cols))


In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras import layers, Model

    def build_bilstm_mha(max_len=200, descriptor_dim=126, vocab_size=22, num_heads=4):
        seq_in = layers.Input(shape=(max_len,), name='sequence_index_input')
        x = layers.Embedding(vocab_size, 64, mask_zero=True, name='aa_embedding')(seq_in)
        x = layers.Bidirectional(layers.LSTM(64, return_sequences=True), name='bilstm')(x)
        attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=32, name='multi_head_attention')(x, x)
        x = layers.GlobalAveragePooling1D(name='attention_pooling')(attn)

        desc_in = layers.Input(shape=(descriptor_dim,), name='gpsd_descriptor_input')
        d = layers.Dense(64, activation='relu', name='gpsd_dense')(desc_in)

        z = layers.Concatenate(name='sequence_gpsd_concat')([x, d])
        z = layers.Dense(64, activation='relu', name='classifier_dense')(z)
        out = layers.Dense(1, activation='sigmoid', name='amp_probability')(z)
        model = Model([seq_in, desc_in], out, name='BiLSTM_MHA')
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model

    model = build_bilstm_mha(descriptor_dim=len(descriptor_cols))
    model.summary()
except ImportError as exc:
    print('Install tensorflow from requirements.txt to build the BiLSTM-MHA model:', exc)


In [ ]:
from make_result_tables import main as make_result_tables
make_result_tables()
print((ROOT / 'results' / 'table4_internal_model_comparison.csv').read_text(encoding='utf-8'))
